# BladesTime — Blades of Time (Switch → iOS 26)

## Só uma célula de execução
Escolha **ETAPA** no seletor da célula abaixo e toque em **▶**. A própria célula autentica o Drive, baixa o jogo, prepara o port, envia o workspace ao Drive e, após solicitar o token, dispara **um** build no Actions.

**PREPARAR + ACTIONS** — use na primeira execução. A base e o update de `botime` são baixados em chunks paralelos de 64 MiB (quando suportado), verificados, extraídos e preparados; RomFS e workspace são enviados em blocos ao Drive.

**RECUPERAR UPLOAD + ACTIONS** — somente se o preparo terminou e os uploads foram concluídos, mas a sessão do Colab caiu antes de disparar o build. Não repete extração.

**TRANSFERIR IPA PARA DRIVE** — somente depois que o Actions terminar com sucesso. Baixa a última IPA verificada da branch e envia a `botime/ipa`, sem tocar nos NSPs.

Você verá cada fase no resultado da célula. O token GitHub é pedido com entrada oculta, jamais cole PAT no notebook.


In [ ]:
# @title BladesTime — selecione a etapa e clique em ▶
ETAPA = "PREPARAR + ACTIONS" # @param ["PREPARAR + ACTIONS", "RECUPERAR UPLOAD + ACTIONS", "TRANSFERIR IPA PARA DRIVE"]
from google.colab import auth
from pathlib import Path
import getpass, os, subprocess, sys

def fase(numero, mensagem):
    print(f"\n{'=' * 12} BladesTime {numero}: {mensagem} {'=' * 12}\n", flush=True)

def executar(*argumentos, env=None):
    subprocess.run([sys.executable, "-u", *map(str, argumentos)], check=True, env=env)

fase("1", "Autenticando seu Google Drive")
auth.authenticate_user()
repo = Path("/content/bladestime-pipeline")
fase("2", "Atualizando somente o código BladesTime; o cache de game data é preservado")
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "BladesTime",
                    "https://github.com/joao9vegeto-stack/KintTany.git", str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", "BladesTime"], check=True)
    subprocess.run(["git", "-C", str(repo), "reset", "--hard", "FETCH_HEAD"], check=True)

script = repo / "blades" / "colab" / "run_colab.py"
parent_id = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"  # Drive configurado; procura botime automaticamente
handoff = Path("/content/bladestime-handoff.json")

if ETAPA == "TRANSFERIR IPA PARA DRIVE":
    fase("3", "Localizando a última IPA do Actions e enviando a botime/ipa")
    token = getpass.getpass("GitHub PAT com Actions: Read (somente KintTany): ").strip()
    if not token:
        raise RuntimeError("Token GitHub não fornecido; nenhum download foi iniciado.")
    ambiente = os.environ.copy()
    ambiente["BLADES_GITHUB_TOKEN"] = token
    del token
    try:
        executar(repo / "blades" / "colab" / "transfer_ipa.py", env=ambiente)
    finally:
        ambiente.pop("BLADES_GITHUB_TOKEN", None)
elif ETAPA in ("PREPARAR + ACTIONS", "RECUPERAR UPLOAD + ACTIONS"):
    if ETAPA == "PREPARAR + ACTIONS":
        fase("3/5", "BAIXANDO base+update, preparando o port e enviando workspace/RomFS ao Drive")
        executar(script, "prepare", "--drive-folder-id", parent_id)
    else:
        fase("3/5", "Recuperando o handoff previamente salvo no Drive, sem refazer o NSP")
        handoff.unlink(missing_ok=True)
        executar(script, "recover", "--drive-folder-id", parent_id)
    if not handoff.is_file():
        raise RuntimeError("Handoff ainda não disponível no Drive. Não é seguro disparar o Actions.")
    fase("4/5", "Upload ao Drive concluído; autorizado a publicar um handoff no GitHub")
    token = getpass.getpass("GitHub PAT Contents: Read/Write (somente KintTany): ").strip()
    if not token:
        raise RuntimeError("Token GitHub não fornecido. O checkpoint no Drive continua salvo.")
    ambiente = os.environ.copy()
    ambiente["BLADES_GITHUB_TOKEN"] = token
    del token
    try:
        executar(script, "dispatch", "--handoff", handoff, env=ambiente)
    finally:
        ambiente.pop("BLADES_GITHUB_TOKEN", None)
    fase("5/5", "Handoff publicado; acompanhe o build na branch BladesTime")
    print("https://github.com/joao9vegeto-stack/KintTany/actions/workflows/bladestime.yml", flush=True)
    print("Depois do build bem-sucedido, escolha TRANSFERIR IPA PARA DRIVE aqui.", flush=True)
else:
    raise ValueError("Etapa desconhecida: " + str(ETAPA))
